In [1]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 8.9 MB/s  0:00:00


In [5]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, Namespace, RDF

# 1. Initialize Graph and Namespace
g = Graph()
IOT = Namespace("http://example.org/iot#")

# 2. Define Ontology Properties
connects_to = IOT.connectsTo
has_protocol = IOT.hasProtocol
has_port = IOT.hasPort
has_threat_label = IOT.hasThreatLabel

# 3. Load the uploaded CSV
df = pd.read_csv('iot23_sample.csv', on_bad_lines='skip')

# 4. Iterate through rows to create RDF triples
for index, row in df.iterrows():
    try:
        src_ip = str(row['id.orig_h']).replace('.', '_')
        dst_ip = str(row['id.resp_h']).replace('.', '_')

        src_node = URIRef(IOT + f"IP_{src_ip}")
        dst_node = URIRef(IOT + f"IP_{dst_ip}")

        g.add((src_node, RDF.type, IOT.Device))
        g.add((dst_node, RDF.type, IOT.NetworkNode))
        g.add((src_node, connects_to, dst_node))

        g.add((dst_node, has_port, Literal(row['id.resp_p'])))
        g.add((dst_node, has_protocol, Literal(row['proto'])))

        label = str(row['label'])
        if label != '-' and label != 'Benign' and pd.notna(label):
            g.add((src_node, has_threat_label, Literal(label)))

    except Exception as e:
        continue

# 5. Export to Turtle format
g.serialize(destination='iot_knowledge_graph.ttl', format='turtle')
print(f"Success! Generated {len(g)} RDF triples and saved to iot_knowledge_graph.ttl")

Success! Generated 800 RDF triples and saved to iot_knowledge_graph.ttl


In [6]:
from google.colab import files
files.download('iot_knowledge_graph.ttl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>